In [1]:
import cv2
import numpy as np
import glob
from pathlib import Path

# 1. Matches your printed checkerboard: 10 internal corners wide, 7 high
CHECKERBOARD = (10, 7) 
subpix_criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 30, 0.1)

# 2. Storage
objpoints = [] # 3D world space
imgpoints = [] # 2D image plane

# 3. Path to your calibration photos
images = glob.glob("/home/wiebe/Desktop/Robotic/AE4317/testing_nn/data/calibration/*.jpg")

for fname in images:
    img = cv2.imread(fname)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    
    # Find corners
    ret, corners = cv2.findChessboardCorners(gray, CHECKERBOARD, 
        cv2.CALIB_CB_ADAPTIVE_THRESH + cv2.CALIB_CB_FAST_CHECK + cv2.CALIB_CB_NORMALIZE_IMAGE)
    
    if ret:
        objp = np.zeros((1, CHECKERBOARD[0] * CHECKERBOARD[1], 3), np.float32)
        objp[0,:,:2] = np.mgrid[0:CHECKERBOARD[0], 0:CHECKERBOARD[1]].T.reshape(-1, 2)
        objpoints.append(objp)
        
        cv2.cornerSubPix(gray, corners, (3, 3), (-1, -1), subpix_criteria)
        imgpoints.append(corners)

# 4. Fisheye Calibration
N_OK = len(imgpoints)
K = np.zeros((3, 3))
D = np.zeros((4, 1))

if N_OK > 0:
    rms, _, _, _, _ = cv2.fisheye.calibrate(
        objpoints, imgpoints, gray.shape[::-1], K, D, None, None,
        cv2.fisheye.CALIBRATE_FIX_SKEW + cv2.fisheye.CALIBRATE_RE_PROJECT_LOG,
        (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 30, 1e-6)
    )
    print(f"K Matrix:\n{K}")
    print(f"D Coeffs:\n{D}")
else:
    print("No corners detected. Check lighting or board flatness.")

No corners detected. Check lighting or board flatness.
